# Export Komposit Citra per Tahun — Time-Series (inferensi, IMAGE-ONLY)

Export komposit median Sentinel-2 **6 band** (tanpa label fusion) untuk tahun-tahun yang akan
dianalisis time-series, sebagai **input inferensi** model 7-kelas. Target akhir: **2022, 2023,
2024** (2021 & 2025 sudah ada; 2026 ditunda karena baru terisi setengah tahun).

**Fokus run ini: HANYA 2023**, lewat project GEE `forestwatch-498909` (akun Google teman tim --
identitas asli berbeda, bukan akun/proyek tambahan demi menyiasati kuota). 2022 & 2024 disusun
terpisah nanti (masing-masing kemungkinan lewat akun/project lain juga, dijalankan oleh
pemiliknya sendiri).

**Penting — kenapa image-only:** label fusion (`build_label`) cuma dibutuhkan saat bikin data
*training*. Untuk inferensi tahun baru, model cuma perlu komposit citra 6 band. Jadi export ini
JAUH lebih ringan dari export training (tanpa Hansen/DW/Mining/Palm overlay).

**Grid tile**: 6x6 = 36 tile, scale 10 m, dari `PAPUA_BBOX` — **identik** dengan export 2021/2025
supaya mask antar-tahun ter-align piksel (wajib untuk change detection).

Alur penuh time-series (notebook ini cuma TAHAP 1, dan baru cakup 2023):
1. **Export komposit citra/tahun** (notebook ini) -> 36 GeoTIFF/tahun ke Drive
2. Inferensi per tahun (`run_inference.py`) -> mask tutupan lahan/tahun
3. Render `landcover_YYYY.png` + statistik/tahun
4. Change detection: year-over-year (laju) + kumulatif-dari-2021 (total)
5. WebGIS: slider tahun / multi-layer


In [ ]:
# === Bagian 0 -- Setup environment (Colab) + auth GEE (akun teman tim, project forestwatch-498909) ===
import os, sys, subprocess
from pathlib import Path

subprocess.run("cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
               "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
               shell=True, check=False)
subprocess.run("pip install -q -e /content/fw_repo[gee]", shell=True, check=False)

_SRC = "/content/fw_repo/model/src"
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

from forestwatch.constants import PAPUA_BBOX, BANDS
from forestwatch.gee.auth import init_ee

# PENTING: jalankan cell ini login sbg akun Google PEMILIK project forestwatch-498909
# (akun teman tim), bukan akun lain -- init_ee akan gagal/ee.Authenticate() minta login
# ulang kalau akun yg aktif tak punya akses ke project ini.
init_ee(project="forestwatch-498909")
import ee

print("GEE siap (project aktif: forestwatch-498909).")
print("Band komposit:", BANDS)
print("PAPUA_BBOX:", PAPUA_BBOX)


In [ ]:
# === Bagian 1 -- Region + grid tile (sekali, project-agnostic, dipakai semua tahun) ===
from forestwatch.gee.tiles import make_tiles

NX, NY, SCALE = 6, 6, 10   # IDENTIK dgn export 2021/2025 -> mask antar-tahun align
region = ee.Geometry.Rectangle(list(PAPUA_BBOX))
tiles = make_tiles(region, nx=NX, ny=NY)
print(f"Grid: {NX}x{NY} = {len(tiles)} tile (scale {SCALE} m).")


## Bagian 2 — Config: export 2023 (project forestwatch-498909)

In [ ]:
# === Config export -- fokus 2023 saja (project forestwatch-498909, akun teman tim) ===
YEARS = [2023]                       # 2022 & 2024 disusun terpisah (akun/project lain)
GEE_PROJECTS = ["forestwatch-498909"]
assert len(YEARS) == len(GEE_PROJECTS), "Jumlah tahun harus pas dgn jumlah project (1:1)."
YEAR_PROJECT = dict(zip(YEARS, GEE_PROJECTS))

DRIVE_FOLDER_TMPL = "ForestWatch_Tiles_{year}"   # subfolder Drive per tahun (Drive akun teman tim)
NAME_PREFIX_TMPL  = "papua_{year}_tile"          # prefix nama file/task per tile
DRY_RUN = True   # True = cuma print rencana (TAK start task). Set False utk export sungguhan.

print("Rencana export:")
for y in YEARS:
    print(f"  {y} -> project '{YEAR_PROJECT[y]}' | {len(tiles)} tile -> Drive folder "
          f"'{DRIVE_FOLDER_TMPL.format(year=y)}' (prefix '{NAME_PREFIX_TMPL.format(year=y)}')")
print(f"\nTotal task = {len(YEARS) * len(tiles)} | DRY_RUN = {DRY_RUN}")
print("Set DRY_RUN = False lalu jalankan ulang cell berikutnya utk mulai export sungguhan.")


In [ ]:
# === Export komposit citra (image-only) per tahun -- switch project GEE per tahun ===
from forestwatch.gee.composite import s2_composite
from forestwatch.gee.export import export_tiles_grid

all_tasks = {}
for y in YEARS:
    project = YEAR_PROJECT[y]
    folder = DRIVE_FOLDER_TMPL.format(year=y)
    prefix = NAME_PREFIX_TMPL.format(year=y)
    if DRY_RUN:
        print(f"[DRY RUN] {y} (project '{project}'): akan export {len(tiles)} tile -> {folder} "
              f"(prefix {prefix})")
        continue
    init_ee(project=project)   # switch project aktif -- task tahun ini kena kuota project ini
    img = s2_composite(y, region)   # 6 band float, reflektansi [0,1] -- TANPA label
    tasks = export_tiles_grid(img, tiles, name_prefix=prefix, folder=folder, scale=SCALE)
    all_tasks[y] = tasks
    print(f"{y}: {len(tasks)} task dimulai di project '{project}' -> Drive '{folder}'")

if not DRY_RUN:
    total = sum(len(t) for t in all_tasks.values())
    print(f"\n{total} task export dimulai (tersebar di {len(set(YEAR_PROJECT.values()))} project GEE).")
    print("Pantau di https://code.earthengine.google.com/tasks (ganti project di pojok kiri atas")
    print("utk lihat task tahun lain), atau jalankan cell monitor di bawah.")
else:
    print("\n(DRY RUN -- tak ada task dimulai, tak ada project di-switch. Set DRY_RUN=False utk export sungguhan.)")


In [ ]:
# === (Opsional) Monitor status task sampai semua selesai (semua project sekaligus) ===
from forestwatch.gee.export import monitor_tasks

if all_tasks:
    flat = [t for tasks in all_tasks.values() for t in tasks]
    print(f"Memantau {len(flat)} task dari {len(all_tasks)} project (poll tiap 60 dtk)...")
    final = monitor_tasks(flat, poll_seconds=60, timeout_seconds=6 * 3600)
    print("Status akhir:", final)
else:
    print("Belum ada task (masih DRY_RUN atau cell export belum dijalankan).")


## Langkah selanjutnya (setelah semua task COMPLETED)

1. **Cek Drive**: tiap folder `ForestWatch_Tiles_{year}` berisi 36 GeoTIFF (semua project export
   ke Drive akun yang sama, jadi tetap satu lokasi meski 3 project GEE berbeda yang memprosesnya).
2. **Inferensi per tahun** -- untuk tiap tahun, jalankan model 7-kelas di folder tile-nya
   (lihat `model/scripts/run_inference.py`). Hasil: mask tutupan lahan per tahun.
3. **Render + change detection** -- setelah mask per tahun jadi, baru kita susun
   `landcover_YYYY.png` + statistik, lalu change detection year-over-year & kumulatif-dari-2021.

> Catatan: notebook ini HANYA export citra (tahap 1). Tahap inferensi (butuh GPU + checkpoint
> `best_model_finetune_v2.pt`) dan change-detection time-series akan disiapkan terpisah.
